# Companion Notebook: Exploratory Data Analysis

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/examples/15_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook follows the content from *Chapter 15: Exploratory Data Analysis*. It walks through a complete EDA of the Complete Journey shopper data — from raw transactions to segmented household profiles to a coherent set of findings — modeling the process a working analyst follows when handed a new dataset and a business question.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from completejourney_py import get_data

cj_data = get_data()
transactions = cj_data['transactions']
demographics = cj_data['demographics']

print(transactions.shape)
print(demographics.shape)

---

## The Question

**How frequently do our shoppers visit, and what drives those patterns?**

Start by orienting to the data: shape, structure, and what each table contains.

In [ ]:
transactions.head()

In [ ]:
transactions.info()

---

## Understanding Shopping Trips

Before asking how *often* shoppers visit, define the unit of analysis: the shopping trip. A trip is identified by `basket_id` — all items purchased together share the same basket. Collapse the item-level transactions to one row per basket.

In [ ]:
trips = (
    transactions
    .assign(trip_date=transactions['transaction_timestamp'].dt.normalize())
    .groupby(['household_id', 'basket_id', 'trip_date'], as_index=False)
    .agg(
        spend=('sales_value', 'sum'),
        items=('quantity', 'sum')
    )
)

trips.shape

In [ ]:
trips.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(trips['spend'].clip(upper=150), bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Trip Spend')
axes[0].set_xlabel('Basket Spend ($, clipped at $150)')
axes[0].set_ylabel('Number of Trips')

sns.histplot(trips['items'].clip(upper=40), bins=40, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Items per Trip')
axes[1].set_xlabel('Items in Basket (clipped at 40)')
axes[1].set_ylabel('')

fig.suptitle('Shopping Trip Characteristics', fontsize=14, fontweight='bold');
plt.tight_layout()

---

## Shopping Frequency

To measure visit frequency, compute the time gap between consecutive trips for each household. Sort by household and date, then use `.diff()` on the date column.

In [ ]:
trips_sorted = trips.sort_values(['household_id', 'trip_date'])

trips_sorted['days_since_last'] = (
    trips_sorted
    .groupby('household_id')['trip_date']
    .diff()
    .dt.days
)

trips_sorted.head()

In [ ]:
trips_sorted['days_since_last'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

inter_visit = trips_sorted['days_since_last'].dropna()

sns.histplot(inter_visit.clip(upper=60), bins=60, kde=True, ax=axes[0])
axes[0].set_title('Days Between Trips (clipped at 60 days)')
axes[0].set_xlabel('Days Since Last Visit')
axes[0].set_ylabel('Number of Trips')

sns.histplot(inter_visit[inter_visit <= 15], bins=15, kde=True, ax=axes[1])
axes[1].set_title('Days Between Trips — Frequent Range')
axes[1].set_xlabel('Days Since Last Visit (0–15 days)')
axes[1].set_ylabel('')

fig.suptitle('Shopping Trip Frequency Distribution', fontsize=14, fontweight='bold');
plt.tight_layout()

---

## Segmenting Shoppers

Aggregate to the household level and classify each household into a frequency tier based on average inter-visit gap.

In [ ]:
household_freq = (
    trips_sorted
    .groupby('household_id')['days_since_last']
    .agg(
        avg_days_between='mean',
        median_days_between='median'
    )
    .reset_index()
    .dropna(subset=['avg_days_between'])
)

household_freq.head()

In [ ]:
household_freq['avg_days_between'].describe()

In [ ]:
def classify_frequency(days):
    if days <= 2:
        return 'Very Frequent'
    elif days <= 7:
        return 'Frequent'
    elif days <= 14:
        return 'Occasional'
    elif days <= 30:
        return 'Infrequent'
    else:
        return 'Rare'

tier_order = ['Very Frequent', 'Frequent', 'Occasional', 'Infrequent', 'Rare']
household_freq['frequency_tier'] = household_freq['avg_days_between'].apply(classify_frequency)

In [ ]:
tier_counts = (
    household_freq['frequency_tier']
    .value_counts()
    .reindex(tier_order)
    .reset_index()
)
tier_counts.columns = ['frequency_tier', 'n_households']

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tier_counts['frequency_tier'], tier_counts['n_households'], color='steelblue')
ax.set_xlabel('Number of Households')
ax.set_title('Shopper Frequency Segments', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for i, row in tier_counts.iterrows():
    ax.text(row['n_households'] + 5, i, str(row['n_households']), va='center', fontsize=9)
plt.tight_layout()

In [ ]:
trip_spend_by_tier = (
    trips_sorted
    .merge(household_freq[['household_id', 'frequency_tier']], on='household_id', how='inner')
)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(
    data=trip_spend_by_tier,
    x='frequency_tier',
    y='spend',
    order=tier_order,
    ax=ax
)
ax.set_ylim(0, 120)
ax.set_xlabel('Frequency Tier')
ax.set_ylabel('Spend per Trip ($)')
ax.set_title('Trip Spend by Shopper Frequency Tier', fontsize=13, fontweight='bold')
plt.tight_layout()

---

## Demographic Patterns

Join the demographics table to ask whether frequency tiers correspond to demographic characteristics.

In [ ]:
demo_freq = household_freq.merge(demographics, on='household_id', how='left')
demo_freq.shape

### Does income predict frequency?

Build a pivot table of household counts by income range × frequency tier and visualize it as a heatmap.

In [ ]:
income_order = [
    'Under 15K', '15-24K', '25-34K', '35-49K',
    '50-74K', '75-99K', '100-124K', '125-149K',
    '150-174K', '175-199K', '200-249K', '250K+'
]

income_heat = (
    demo_freq
    .dropna(subset=['income', 'frequency_tier'])
    .groupby(['income', 'frequency_tier'])
    .size()
    .reset_index(name='count')
    .pivot(index='income', columns='frequency_tier', values='count')
    .fillna(0)
    .reindex(income_order)
    .reindex(columns=tier_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    income_heat,
    cmap='YlOrRd',
    ax=ax,
    annot=True,
    fmt='.0f',
    linewidths=0.4,
    cbar_kws={'label': 'Number of Households'}
)
ax.set_xlabel('Frequency Tier')
ax.set_ylabel('')
ax.set_title('Household Count by Income Range × Frequency Tier', fontsize=13, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()

### Does household size matter?

Compare the average inter-visit gap across household sizes using a boxplot with explicit ordering.

In [ ]:
size_order = ['1', '2', '3', '4', '5+']

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(
    data=demo_freq.dropna(subset=['household_size', 'avg_days_between']),
    x='household_size',
    y='avg_days_between',
    order=size_order,
    ax=ax
)
ax.set_ylim(0, 20)
ax.set_xlabel('Household Size')
ax.set_ylabel('Avg Days Between Trips')
ax.set_title('Shopping Frequency by Household Size', fontsize=13, fontweight='bold')
plt.tight_layout()

---

## Weekly Shopping Patterns

Does shopping behavior vary by day of the week, and does that pattern differ across frequency tiers?

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

trips_with_tier = (
    trips_sorted
    .assign(day_of_week=lambda x: x['trip_date'].dt.day_name())
    .merge(household_freq[['household_id', 'frequency_tier']], on='household_id', how='inner')
)

day_tier_counts = (
    trips_with_tier
    .groupby(['frequency_tier', 'day_of_week'])
    .size()
    .reset_index(name='n_trips')
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(
    data=day_tier_counts,
    x='day_of_week',
    y='n_trips',
    hue='frequency_tier',
    order=day_order,
    hue_order=tier_order,
    ax=ax
)
ax.set_xlabel('')
ax.set_ylabel('Number of Trips')
ax.set_title('Trips by Day of Week and Frequency Tier', fontsize=13, fontweight='bold')
ax.legend(title='Frequency Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

### Time of Day

Do frequency tiers differ in *when* during the day they shop? Recover the hour from the original transactions (since `trip_date` was normalized to midnight), then normalize within each tier so tiers of different sizes are comparable.

In [ ]:
basket_hour = (
    transactions
    .groupby('basket_id')['transaction_timestamp']
    .first()
    .dt.hour
    .reset_index()
    .rename(columns={'transaction_timestamp': 'hour_of_day'})
)

hour_tier = (
    trips_sorted[['basket_id', 'household_id']]
    .drop_duplicates()
    .merge(basket_hour, on='basket_id')
    .merge(household_freq[['household_id', 'frequency_tier']], on='household_id')
    .groupby(['frequency_tier', 'hour_of_day'])
    .size()
    .reset_index(name='n_trips')
)

hour_tier['pct_trips'] = (
    hour_tier
    .groupby('frequency_tier')['n_trips']
    .transform(lambda x: x / x.sum() * 100)
)

fig, ax = plt.subplots(figsize=(12, 5))
for tier in tier_order:
    data = hour_tier[hour_tier['frequency_tier'] == tier]
    ax.plot(data['hour_of_day'], data['pct_trips'], marker='o', markersize=4, label=tier)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Share of Trips (%)')
ax.set_title('Time-of-Day Shopping Patterns by Frequency Tier', fontsize=13, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.legend(title='Frequency Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

---

## Telling the Story

A good EDA ends with a narrative, not just code output. Here is a summary of the key findings from this analysis:

**Key Findings:**

- **There is no single "typical" shopper.** Frequency is highly variable — most households fall into the Frequent tier (every 3–7 days), but a meaningful tail of Infrequent and Rare visitors exists. Any marketing strategy that treats the shopper base as homogeneous is leaving value on the table.

- **Spending per trip follows a loyalty-shaped curve.** Median trip spend *increases* from Very Frequent → Occasional shoppers, then *drops* for Infrequent and Rare visitors. Loyal customers consolidate staple purchases into larger baskets; low-frequency visitors make smaller, purpose-driven trips with no brand affinity.

- **Demographics do not reliably explain frequency.** Income does not predict tier membership — "Frequent" dominates every income bracket. Household size shows no consistent trend either. Critically, Infrequent and Rare shoppers are almost entirely absent from the demographics table, suggesting they never enrolled in the loyalty program — itself a signal of low store engagement.

- **Weekend traffic is a broad-based surge, not a segment effect.** All frequency tiers see elevated trip counts on Saturday and Sunday. Weekend staffing decisions should be driven by overall volume, not assumptions about who is shopping.

- **All segments shop at the same time of day.** Every frequency tier follows an identical time-of-day curve: near-zero between 5–9am, building through the day, peaking at 9–10pm. Time-targeted promotions would not produce different outcomes across frequency segments.

**What Would We Investigate Next?**

1. Cumulative annual value by segment — do Very Frequent shoppers (less per trip) actually accumulate more annual spend than Occasional shoppers (more per trip)?
2. Basket composition by tier — what are different frequency groups actually buying, and what does that mean for product assortment and promotions?
3. Loyalty trajectory over time — are Infrequent shoppers declining (churning) or stable?
4. Promotional responsiveness of low-frequency segments — can targeted offers convert Infrequent visitors into Occasional shoppers?